This notebook contiains the computations from the examples of the paper "Copositivity, discriminants and nonseparable signed supports" by Elisenda Feliu, Joan Ferrer and Máté L. Telek.


In [1]:
using Oscar;
using CopositivityDiscriminants;
using HomotopyContinuation; 
const HC = HomotopyContinuation; 

  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.5.0


[ Info: Precompiling CopositivityDiscriminants [bba005fc-0f3c-480a-8e42-a26130d8489d] (cache misses: include_dependency fsize change (2), wrong dep version loaded (4), mismatched flags (12))


## Example 2.11

In [2]:
R , (x,y,t,c0,c1,c2,c3,c4) = polynomial_ring(QQ,["x","y","t","c0","c1","c2","c3","c4"])

f=c0+c1*x^2+c2*y^2+c3*x^2*y^2-c4*t*x*y
fx=2*c1*x^2+2*c3*x^2*y^2-c4*t*x*y
fy=2*c2*y^2+2*c3*x^2*y^2-c4*t*x*y

I=ideal([f,fx,fy])
I2=I:ideal([c0*c1*c3*c4*x*y]) #we saturate to avoid solutions where some variable is zero
eliminate(I2,[x,y])

Ideal generated by
  t^4*c4^4 - 8*t^2*c0*c3*c4^2 - 8*t^2*c1*c2*c4^2 + 16*c0^2*c3^2 - 32*c0*c1*c2*c3 + 16*c1^2*c2^2

## Example 4.1

In [3]:
HC.@var x[1:2] t

gt=49-56.056*t*x[1]-56.056*t*x[2]+32*x[1]*x[2]-8.008*t*x[1]*x[2]^2-8.008*t*x[1]^2*x[2]+2*x[1]^2*x[2]^2+30*x[1]^2-8.008*t*x[1]^3+x[1]^4+30*x[2]^2-8.008*t*x[2]^3+x[2]^4

gtx1=differentiate(gt,x[1])
gtx2=differentiate(gt,x[2])

F=System([gt,x[1]*gtx1,x[2]*gtx2],variables=[t,x[1],x[2]])

sol=HC.solve(F)


Tracking 36 paths... 100%|██████████████████████████████| Time: 0:00:04
                   # paths tracked: 36
   # non-singular solutions (real): 6 (6)
       # singular endpoints (real): 30 (0)
          # total solutions (real): 36 (6)


Result with 36 solutions
• 36 paths tracked
• 6 non-singular solutions (6 real)
• 30 singular solutions (0 real)
• random_seed: 0x1d2a4836
• start_system: :polyhedral
• multiplicity table of singular solutions:
╭───────┬───────┬────────┬────────────╮
│ mult. │ total │ # real │ # non-real │
├───────┼───────┼────────┼────────────┤
│   1   │  30   │   0    │     30     │
╰───────┴───────┴────────┴────────────╯


In [4]:
cert=certificates(HC.certify(F,sol))
for c in cert
    if HC.is_positive(c)
        println(solution_candidate(c))
    end
end

Certifying 6 solutions... 100%|█████████████████████████| Time: 0:00:00
          # processed: 6
   # certified (real): 6 (6)
    # distinct (real): 6 (6)
ComplexF64[1.0012284287428486 + 6.240157223946451e-46im, 1.8708286933869696 - 3.117889083122718e-44im, 1.8708286933869749 + 5.885453550164232e-44im]


## Example 4.7

In [5]:
HC.@var x[1:2] t c[1:5];

ft= c[1] +
       c[2]*x[1]^2 +
       c[3]*x[2]^2 +
       c[4]*x[1]^2*x[2]^2 -
       c[5]*t*x[1]*x[2]   

F=System([ft,x[1]*differentiate(ft,x[1]),x[2]*differentiate(ft,x[2])],parameters=c,variables=[t,x[1],x[2]]);

start_solutions = [[1,1,1]]
c_target=[1,1,1,1,1]; 
c_start=[1/4,1/4,1/4,1/4,1]; # [1/4,...,1/4] are affine coordinates of x[1]*x[2] with respect to the other monomials
resc=HC.solve(F,start_solutions; start_parameters=c_start, target_parameters=c_target);
solutions(resc)

1-element Vector{Vector{ComplexF64}}:
 [4.0 + 0.0im, 1.0 + 0.0im, 1.0 + 0.0im]

## Example 4.8

In [9]:
HC.@var x[1:4]

d=(10/9)^(9/10)*40^(1/10); #circuit number
e=10^(-7) #perturbation
f=1+x[1]^40+x[2]^40+x[3]^40+x[4]^40-(d+e)*x[1]*x[2]*x[3]*x[4]


t0 = time_ns()
r = check_copositivity(f)   
dt = (time_ns() - t0) * 1e-9 #convert to seconds
min=dt/60


println("The polynomial is copositive: $(r.copositive).")
println("Certified interval for t_min:  $(r.t_min_interval).")
println("Computed in approximately $(min) minutes." )



Certifying 2560000 solutions... 100%|███████████████████| Time: 0:01:17
          # processed: 2560000
   # certified (real): 2560000 (16)
    # distinct (real): 2560000 (16)
The polynomial is copositive: false.
Certified interval for t_min:  [0.99999993710556 +/- 6.57e-15] + [+/- 2.39e-23]im.
Computed in approximately 11.769745584733334 minutes.


In [11]:
#Now we use the nonseparable signed support structure
t0 = time_ns()
r = check_copositivity(f,nonseparable=true)   
dt = (time_ns() - t0) * 1e-9 #convert to seconds


println("The polynomial is copositive: $(r.copositive).")
println("Certified interval for t_min:  $(r.t_min_interval).")
println("Computed in approximately $(dt) seconds." )

The polynomial is copositive: false.
Certified interval for t_min:  [0.99999993710556 +/- 6.57e-15] + [+/- 2.39e-23]im.
Computed in approximately 5.152437375000001 seconds.


In [12]:
d=(10/9)^(9/10)*40^(1/10); #circuit number

for i in 8:15
    e=10.0^(-i) #perturbation
    f=1+x[1]^40+x[2]^40+x[3]^40+x[4]^40-(d+e)*x[1]*x[2]*x[3]*x[4]
    r=check_copositivity(f,nonseparable=true)
    println(r.copositive)
end

false
false
false
false
false
false
false
missing


`check_copositivity`correctly certifies that $f$ attains negative values up to perturbation of the order $10^{-15}$, where it cannot certify it.


In [13]:
# Evaluating crtical points method.

d = (10/9)^(9/10) * 40^(1/10)
e = 10.0^(-7)
f = 1 + x[1]^40 + x[2]^40 + x[3]^40 + x[4]^40 - (d+e) * (x[1]*x[2]*x[3]*x[4])

t0 = time_ns()

F = System([differentiate(f,x[i]) for i in 1:4]; variables=[x[1], x[2], x[3], x[4]]) #compute critical points
result = HomotopyContinuation.solve(F)

C = certify(F, result)
certs = certificates(C)

pos = [c for c in certs if HomotopyContinuation.is_positive(c)] # Keep only positive certificates

cand=solution_candidate.(pos)
sol=real.(cand)

for s in sol
    ev=HC.evaluate(f,[x[1], x[2], x[3], x[4]]=>s)
    println("Positive critical point:  $(s),  Evaluation:  $(ev)")
end
dt = (time_ns() - t0) * 1e-9 #convert to seconds
min= dt/60
println("Time: $(min) minutes.")

Certifying 2304000 solutions... 100%|███████████████████| Time: 0:00:53
          # processed: 2304000
   # certified (real): 2304000 (8)
    # distinct (real): 2304000 (8)
Positive critical point:  [0.9143078283591858, 0.9143078283591858, 0.9143078283591858, 0.9143078283591858],  Evaluation:  -6.988271224889209e-8
Time: 7.197833809033333 minutes.
